 # XGBoost for exceedance probability of hourly avg (>30 ppb)

## Load libraries

In [71]:
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import LeaveOneGroupOut, GroupKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    mean_pinball_loss,
)
import joblib
from itertools import product
import time
from sklearn.model_selection import 

## Prepare data set

In [72]:
# Load dataset
save_path = "../rfiles/qxgb_files/"
file_path = "../rfiles/data/"
ha_full_full = pd.read_parquet(file_path + "ha_full_full_for_py_v2.parquet")
ha_full_full_y = pd.read_parquet(file_path + "ha_full_full_y_for_py_v2.parquet")
monitor_coord_map = pd.read_parquet(save_path +'monitor_coord_map.parquet')

In [73]:
hourly_predictors = ['year',
                     'month_sin', 'month_cos',
                     'hour_sin', 'hour_cos',
                     'weekday_sin', 'weekday_cos',
                     'julian_sin', 'julian_cos',
                     'wd_sin', 'wd_cos', 'ws_avg', 
                     'hourly_downwind_ref', 'dist_wrp', 'dist_ref',
                     'mon_utm_x', 'mon_utm_y', 'monthly_oil_2km', 'monthly_gas_2km', 
                     'active_2km', 'inactive_2km', 'hourly_downwind_wrp', 
                     'elevation', 'EVI', 'num_odor_complaints',
                     'dist_dc', 'closest_wrp_capacity', 'hourly_temp', 'hourly_hum', 'hourly_precip']

response = "H2S_hourly_avg"
time_col = "dayhour"
THRESH_PPB = 30.0
THRESH_LOG = np.log(THRESH_PPB)

df = ha_full_full.copy()

# site_id: use coords; if you have a Monitor/site column, use that instead
df["site_id"] = df[["mon_utm_x", "mon_utm_y"]].astype(str).agg("_".join, axis=1)
df = df.merge(monitor_coord_map, on=["mon_utm_x", "mon_utm_y"], how="left")
# Targets
df["y"] = ha_full_full_y[response]
df["y_exc"] = (df["y"] > THRESH_PPB).astype(int)
df["z"] = np.log(df["y"])

# Features
X_all = df[hourly_predictors]
y_raw = df["y"].to_numpy()
y_exc = df["y_exc"].to_numpy()
groups = df["site_id"].to_numpy()

In [4]:
df.head()

,year,month_sin,month_cos,hour_sin,hour_cos,weekday_sin,weekday_cos,julian_sin,julian_cos,wd_sin,...,hourly_temp,hourly_hum,hourly_precip,day,dayhour,site_id,Monitor,y,y_exc,z
0,2021.0,-0.866025,0.5,0.500000,-0.866025,-0.974928,-0.222521,-0.97486,0.22282,0.707107,...,63.4,47,0.0,2021-10-14 00:00:00-07:00,2021-10-14 10,383547.1749268095_3744771.0861730943,Chico,114.987778,1,4.744826
1,2021.0,-0.866025,0.5,0.258819,-0.965926,-0.974928,-0.222521,-0.97486,0.22282,0.224951,...,69.7,28,0.0,2021-10-14 00:00:00-07:00,2021-10-14 11,383547.1749268095_3744771.0861730943,Chico,131.718000,1,4.880663
2,2021.0,-0.866025,0.5,-0.500000,-0.866025,-0.974928,-0.222521,-0.97486,0.22282,-0.766044,...,78.3,21,0.0,2021-10-14 00:00:00-07:00,2021-10-14 14,383547.1749268095_3744771.0861730943,Chico,317.979167,1,5.761986
3,2021.0,-0.866025,0.5,-0.707107,-0.707107,-0.974928,-0.222521,-0.97486,0.22282,-0.866025,...,77.9,22,0.0,2021-10-14 00:00:00-07:00,2021-10-14 15,383547.1749268095_3744771.0861730943,Chico,269.660000,1,5.597162
4,2021.0,-0.866025,0.5,-0.866025,-0.500000,-0.974928,-0.222521,-0.97486,0.22282,-0.984808,...,75.1,29,0.0,2021-10-14 00:00:00-07:00,2021-10-14 16,383547.1749268095_3744771.0861730943,Chico,290.639167,1,5.672083


### Get best hyperparameter tuning from earlier CV

In [5]:
# ---------- Get best params from earlier result ----------
result = joblib.load("ha_q99xgb_tuning_result_v2.pkl")
best_row_old = result["best_row"]
print(best_row_old) 

{'n_estimators': 700.0, 'max_leaves': 128.0, 'eta': 0.2, 'gamma': 0.01, 'colsample_bytree': 0.6, 'min_child_weight': 5.0, 'subsample': 0.75, 'best_iteration': 688.0, 'cv_avg_pinball': 0.014511}


In [6]:
# Define a function that creates a closure to pass the quantile parameter
def pinball_loss_sklearn(preds, dtrain):
    y = dtrain.get_label()
    w = dtrain.get_weight()
    n = y.shape[0]
    K = len(alphas)
    # Handle both shapes returned by XGBoost: (n*K,) or (n, K)
    if preds.ndim == 1:
        if preds.size != n * K:
            raise ValueError(f"preds.size={preds.size} != n*K={n*K}")
        P = preds.reshape(n, K)
    else:
        if preds.shape != (n, K):
            raise ValueError(f"preds shape {preds.shape} != (n={n}, K={K})")
        P = preds
    # Compute pinball per quantile with its own alpha; average them
    losses = [
        mean_pinball_loss(y, P[:, j], alpha=alphas[j],
                          sample_weight=(w if w.size else None))
        for j in range(K)
    ]
    return "avg_pinball", float(np.mean(losses))

In [7]:
def build_xgb_params_from_best_row(best_row, *, objective, eval_metric, extra=None):
    """
    Map your tuned params into xgboost sklearn API params.
    - eta -> learning_rate
    - use max_leaves with grow_policy='lossguide' (best match for max_leaves tuning)
    """

    params = dict(
        max_leaves=int(best_row["max_leaves"]),
        learning_rate=float(best_row["eta"]),
        gamma=float(best_row["gamma"]),
        colsample_bytree=float(best_row["colsample_bytree"]),
        min_child_weight=float(best_row["min_child_weight"]),
        subsample=float(best_row["subsample"]),
        tree_method="hist",
        grow_policy="lossguide",
        objective=objective,
        random_state=42,
    )
    
    if "n_estimators" in best_row:
        params.update(dict(n_estimators = int(best_row["n_estimators"])))
    elif "mean_best_iteration" in best_row:
        params.update(dict(n_estimators = round(best_row["mean_best_iteration"])))

    if "reg_lambda" in best_row:
        params.update(dict(reg_lambda = int(best_row["reg_lambda"])))
                           
    if objective == "reg:quantileerror":
        params.update(dict(custom_metric = eval_metric))
    elif objective == "binary:logistic":
        params.update(dict(eval_metric = eval_metric))
        
    if extra:
        params.update(extra)
    return params

In [8]:
# params for binary exceedance model
bin_params_old = build_xgb_params_from_best_row(
    best_row_old,
    objective="binary:logistic",
    eval_metric="logloss"
)

# params for quantile model (pinball loss)
# NOTE: objective will be set in the regressor constructor; eval_metric can be omitted or set.
q_params_old = build_xgb_params_from_best_row(
    best_row_old,
    objective="reg:quantileerror",
    eval_metric=pinball_loss_sklearn
)

In [9]:
bin_params_old

{'max_leaves': 128,
 'learning_rate': 0.2,
 'gamma': 0.01,
 'colsample_bytree': 0.6,
 'min_child_weight': 5.0,
 'subsample': 0.75,
 'tree_method': 'hist',
 'grow_policy': 'lossguide',
 'objective': 'binary:logistic',
 'random_state': 42,
 'n_estimators': 700,
 'eval_metric': 'logloss'}

In [10]:
q_params_old

{'max_leaves': 128,
 'learning_rate': 0.2,
 'gamma': 0.01,
 'colsample_bytree': 0.6,
 'min_child_weight': 5.0,
 'subsample': 0.75,
 'tree_method': 'hist',
 'grow_policy': 'lossguide',
 'objective': 'reg:quantileerror',
 'random_state': 42,
 'n_estimators': 700,
 'custom_metric': <function __main__.pinball_loss_sklearn(preds, dtrain)>}

## Prepare helper functions

In [11]:
# ------------------------------------------------------------
#  Lag helpers
#  lag(t) = mean z across ALL training sites at time t
# ------------------------------------------------------------
def add_lag_columns_train_log(df_train, *, time_col, z_col="z"):
    """lag(t) = mean log(y) across ALL training sites at time t (same lag for rows at t)."""
    mean_by_time = df_train.groupby(time_col)[z_col].mean()
    out = df_train.copy()
    out["lag"] = out[time_col].map(mean_by_time)   # lag is on log scale
    return out

def add_lag_column_test_log(df_test, df_train, *, time_col, z_col="z"):
    """Apply lag(t) from training sites to held-out site rows (log scale)."""
    mean_by_time = df_train.groupby(time_col)[z_col].mean()
    out = df_test.copy()
    out["lag"] = out[time_col].map(mean_by_time)   # lag is on log scale
    return out

In [12]:
def _to_mat(pred, n, K):
    pred = np.asarray(pred)
    if pred.ndim == 1:
        if pred.size != n * K:
            raise ValueError(f"pred.size={pred.size} != n*K={n*K}")
        return pred.reshape(n, K)
    if pred.shape != (n, K):
        raise ValueError(f"pred shape {pred.shape} != (n={n}, K={K})")
    return pred

def monotone_in_p(q_mat):
    """Enforce nondecreasing quantiles in p (simple rearrangement via cumulative max)."""
    return np.maximum.accumulate(np.asarray(q_mat, dtype=float), axis=1)

def exceed_prob_from_quantiles(q_mat, ps, threshold):
    """
    Convert predicted quantiles Q(p) into P(Y > threshold) by inverting Q(p)=threshold.
    Piecewise-linear inversion in (p, q).
    """
    ps = np.asarray(ps, dtype=float)
    q = monotone_in_p(q_mat)
    n, K = q.shape
    thr = float(threshold)

    # insertion point: first index where q > thr
    idx = np.sum(q <= thr, axis=1)        # in [0..K]
    idx = np.clip(idx, 1, K - 1)          # force bracketing
    j0 = idx - 1
    j1 = idx

    q0 = q[np.arange(n), j0]
    q1 = q[np.arange(n), j1]
    p0 = ps[j0]
    p1 = ps[j1]

    denom = np.where(np.abs(q1 - q0) < 1e-12, 1.0, (q1 - q0))
    frac = np.clip((thr - q0) / denom, 0.0, 1.0)
    p_star = p0 + (p1 - p0) * frac   # approx F(threshold)
    exc = 1.0 - p_star

    # out-of-range handling
    below = thr <= q[:, 0]
    above = thr >= q[:, -1]
    exc[below] = 1.0 - ps[0]
    exc[above] = 1.0 - ps[-1]
    return np.clip(exc, 0.0, 1.0)

## New tuning

### Binary classifier

In [14]:
tune_grid = pd.DataFrame(list(product(
    [64, 128, 256],               # max_leaves
    [0.05, 0.1, 0.2],               # eta
    [0.01, 0.1, 1],            # gamma
    [0.25, 0.5, 0.75],          # colsample_bytree
    [1.0, 5.0, 10],          # min_child_weight
    [0.25, 0.5, 0.75],         # subsample
    [5, 20, 50]    # L2
)), columns=[
    "max_leaves", "eta", "gamma", "colsample_bytree", "min_child_weight", "subsample", "reg_lambda"
])

In [15]:
tune_grid2 = tune_grid.sample(n=300, random_state=42).reset_index(drop=True)

In [16]:
# ============================================================
# 1) Make "leave 3 monitors out" folds (5 folds for 15 monitors)
#    Greedy-balance folds by exceedance count because exceedance is rare.
# ============================================================
def make_3_monitor_folds_balanced(df, group_col="Monitor", y_exc_col="y_exc", n_folds=5, seed=42):
    rng = np.random.default_rng(seed)

    exc = df.groupby(group_col)[y_exc_col].sum().sort_values(ascending=False)
    monitors = exc.index.to_list()

    # Shuffle within equal-count blocks to avoid deterministic ties
    ordered = []
    for c in sorted(exc.unique(), reverse=True):
        block = [m for m in monitors if exc[m] == c]
        rng.shuffle(block)
        ordered.extend(block)

    fold_monitors = [[] for _ in range(n_folds)]
    fold_exc = np.zeros(n_folds, dtype=float)

    for m in ordered:
        j = int(np.argmin(fold_exc))
        fold_monitors[j].append(m)
        fold_exc[j] += float(exc[m])

    folds = []
    all_idx = np.arange(len(df))
    for j in range(n_folds):
        test_mons = set(fold_monitors[j])
        test_mask = df[group_col].isin(test_mons).to_numpy()
        te = all_idx[test_mask]
        tr = all_idx[~test_mask]
        folds.append((tr, te))

    return folds, fold_monitors, fold_exc


# ============================================================
# 2) CV tuning with early stopping (NO LAG for tuning)
#    - Use n_estimators=2000
#    - Early stopping chooses best_iteration per fold
#    - Selection metric: mean PR-AUC (average precision) across folds
# ============================================================
def tune_xgb_exceedance_groupcv_earlystop(
    df,
    predictors,
    tune_grid,
    *,
    group_col="Monitor",
    y_exc_col="y_exc",
    n_folds=5,
    seed=42,
    n_estimators=2000,
    early_stopping_rounds=50,
    verbose_every=25,
):
    folds, fold_mons, fold_exc = make_3_monitor_folds_balanced(
        df, group_col=group_col, y_exc_col=y_exc_col, n_folds=n_folds, seed=seed
    )

    results = []
    for gi, row in tune_grid.iterrows():
        base_params = dict(
            n_estimators=int(n_estimators),
            max_leaves=int(row["max_leaves"]),
            learning_rate=float(row["eta"]),
            gamma=float(row["gamma"]),
            colsample_bytree=float(row["colsample_bytree"]),
            min_child_weight=float(row["min_child_weight"]),
            subsample=float(row["subsample"]),
            reg_lambda=float(row["reg_lambda"]),
            tree_method="hist",
            grow_policy="lossguide",
            max_depth=0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        )

        fold_ap, fold_ll, fold_bs, fold_best_iter = [], [], [], []
        fold_posrate, fold_n = [], []

        for (tr, te) in folds:
            dtr = df.iloc[tr]
            dte = df.iloc[te]

            Xtr = dtr[predictors]
            Xte = dte[predictors]
            ytr = dtr[y_exc_col].to_numpy(dtype=int)
            yte = dte[y_exc_col].to_numpy(dtype=int)

            # per-fold imbalance weight
            pos = int(ytr.sum())
            neg = int(len(ytr) - pos)
            spw = neg / max(pos, 1)

            clf = xgb.XGBClassifier(**base_params, scale_pos_weight=spw, early_stopping_rounds = early_stopping_rounds)

            # Early stopping uses the held-out 3-monitor fold as eval_set
            clf.fit(
                Xtr, ytr,
                eval_set=[(Xte, yte)],
                verbose=False
            )

            p = clf.predict_proba(Xte)[:, 1]

            fold_ap.append(float(average_precision_score(yte, p)))
            fold_ll.append(float(log_loss(yte, p, labels=[0, 1])))
            fold_bs.append(float(brier_score_loss(yte, p)))
            fold_best_iter.append(int(getattr(clf, "best_iteration", np.nan)))
            fold_posrate.append(float(yte.mean()))
            fold_n.append(int(len(yte)))

        out = dict(
            grid_id=int(gi),
            mean_pr_auc=float(np.mean(fold_ap)),
            mean_logloss=float(np.mean(fold_ll)),
            mean_brier=float(np.mean(fold_bs)),
            mean_best_iteration=float(np.nanmean(fold_best_iter)),
            std_pr_auc=float(np.std(fold_ap)),
            std_logloss=float(np.std(fold_ll)),
            std_brier=float(np.std(fold_bs)),
        )
        out.update({k: float(v) for k, v in row.to_dict().items()})
        results.append(out)

        if verbose_every and ((gi + 1) % verbose_every == 0):
            print(f"[{gi+1}/{len(tune_grid)}] PR-AUC={out['mean_pr_auc']:.6f} "
                  f"logloss={out['mean_logloss']:.6f} brier={out['mean_brier']:.6f} "
                  f"best_iter~{out['mean_best_iteration']:.1f}")

    results_df = pd.DataFrame(results)

    # Select best: lowest brier
    results_df = results_df.sort_values(
        ["mean_brier", "mean_logloss"],
        ascending=[True, True]
    ).reset_index(drop=True)

    best_row = results_df.iloc[0].to_dict()
    return best_row, results_df

In [112]:
# ============================================================
# 3) Run tuning
# ============================================================
# start_time = time.perf_counter()
# best_row, cv_results = tune_xgb_exceedance_groupcv_earlystop(
#     df=df,
#     predictors=hourly_predictors,
#     tune_grid=tune_grid2,
#     group_col="Monitor",
#     y_exc_col="y_exc",
#     n_folds=5,                 # 15 monitors -> 5 folds -> ~3 monitors held out each fold
#     seed=42,
#     n_estimators=2000,
#     early_stopping_rounds=50,
#     verbose_every=1,
# )
# end_time = time.perf_counter()
# execution_time = end_time - start_time
# print(f"Execution time: {execution_time:.4f} seconds")
# 
# Save results
# cv_results.toparquet(save_path + "bin_cv_results.parquet", index=False)

[1/300] PR-AUC=0.343946 logloss=0.083154 brier=0.025164 best_iter~306.4
[2/300] PR-AUC=0.358458 logloss=0.072707 brier=0.022291 best_iter~254.6
[3/300] PR-AUC=0.377673 logloss=0.067847 brier=0.020314 best_iter~634.2
[4/300] PR-AUC=0.375808 logloss=0.066793 brier=0.020580 best_iter~507.0
[5/300] PR-AUC=0.354272 logloss=0.074391 brier=0.022128 best_iter~270.2
[6/300] PR-AUC=0.370385 logloss=0.085400 brier=0.026204 best_iter~118.0
[7/300] PR-AUC=0.403828 logloss=0.074102 brier=0.022320 best_iter~172.0
[8/300] PR-AUC=0.374926 logloss=0.072115 brier=0.021647 best_iter~179.2
[9/300] PR-AUC=0.403207 logloss=0.064013 brier=0.018942 best_iter~202.0
[10/300] PR-AUC=0.378612 logloss=0.073619 brier=0.021477 best_iter~210.0
[11/300] PR-AUC=0.363047 logloss=0.081030 brier=0.024431 best_iter~377.4
[12/300] PR-AUC=0.406566 logloss=0.068505 brier=0.020474 best_iter~844.0
[13/300] PR-AUC=0.370441 logloss=0.073135 brier=0.022169 best_iter~237.4
[14/300] PR-AUC=0.319662 logloss=0.084545 brier=0.025866 bes

3284.6101 seconds

About 15 sec for each set of predictors => 15*300/60 = approximately 75 minutes to tune

In [13]:
bin_cv_results = pd.read_parquet(save_path +"bin_cv_results.parquet")

best_row_bin = bin_cv_results.iloc[0].to_dict()
print("\nBest row (by mean brier, then logloss):")
print(best_row_bin)


Best row (by mean brier, then logloss):
{'grid_id': 281.0, 'mean_pr_auc': 0.4016509831496101, 'mean_logloss': 0.06392693003763082, 'mean_brier': 0.0188865104596139, 'mean_best_iteration': 203.8, 'std_pr_auc': 0.21969494907353188, 'std_logloss': 0.12082059088438758, 'std_brier': 0.03589856971986292, 'max_leaves': 64.0, 'eta': 0.2, 'gamma': 0.1, 'colsample_bytree': 0.5, 'min_child_weight': 5.0, 'subsample': 0.75, 'reg_lambda': 50.0}


In [18]:
bin_cv_results

,grid_id,mean_pr_auc,mean_logloss,mean_brier,mean_best_iteration,std_pr_auc,std_logloss,std_brier,max_leaves,eta,gamma,colsample_bytree,min_child_weight,subsample,reg_lambda
0,281,0.401651,0.063927,0.018887,203.8,0.219695,0.120821,0.035899,64.0,0.2,0.10,0.50,5.0,0.75,50.0
1,45,0.424906,0.064079,0.018916,186.0,0.213916,0.120749,0.035884,128.0,0.2,1.00,0.50,5.0,0.75,50.0
2,8,0.403207,0.064013,0.018942,202.0,0.232628,0.121137,0.036014,256.0,0.2,1.00,0.50,1.0,0.75,50.0
3,126,0.413488,0.065276,0.019048,392.2,0.236101,0.121179,0.035948,256.0,0.1,1.00,0.50,10.0,0.75,50.0
4,91,0.415252,0.065765,0.019292,359.0,0.211895,0.122448,0.036422,64.0,0.1,0.10,0.50,5.0,0.75,50.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,145,0.315094,0.093149,0.027799,91.0,0.171663,0.171136,0.051071,256.0,0.2,1.00,0.25,1.0,0.25,5.0
296,194,0.294477,0.092830,0.027879,68.6,0.226674,0.158432,0.048598,256.0,0.2,1.00,0.75,1.0,0.50,5.0
297,274,0.340016,0.093794,0.028030,95.4,0.192250,0.158736,0.048924,256.0,0.2,0.10,0.75,5.0,0.50,5.0
298,40,0.347123,0.094930,0.028162,83.6,0.185780,0.158467,0.048882,64.0,0.2,0.01,0.75,5.0,0.50,5.0


### QXGB

In [19]:
# ============================================================
# QXGB tuning with early stopping (NO lag for tuning)
# Scoring modes:
#   - "pinball": average pinball loss across quantiles (lower better)
#   - "exceed_logloss": logloss of implied exceedance prob at 30 (lower better)
#   - "exceed_brier": brier of implied exceedance prob at 30 (lower better)
#   - "exceed_pr_auc": PR-AUC of implied exceedance prob at 30 (higher better)
# ============================================================
def tune_qxgb_groupcv_earlystop(
    df,
    predictors,
    tune_grid,
    alphas,
    *,
    response_col="y",
    group_col="Monitor",
    y_exc_col="y_exc",
    n_estimators=2000,
    early_stopping_rounds=100,
    n_folds=5,
    seed=42,
    score_mode="pinball",
    verbose_every=25,
):
    folds, fold_mons, fold_exc = make_3_monitor_folds_balanced(
        df, group_col=group_col, y_exc_col=y_exc_col, n_folds=n_folds, seed=seed
    )

    results = []
    K = len(alphas)

    for gi, row in tune_grid.iterrows():
        base_params = dict(
            n_estimators=int(n_estimators),
            max_leaves=int(row["max_leaves"]),
            learning_rate=float(row["eta"]),
            gamma=float(row["gamma"]),
            colsample_bytree=float(row["colsample_bytree"]),
            min_child_weight=float(row["min_child_weight"]),
            subsample=float(row["subsample"]),
            reg_lambda=float(row.get("reg_lambda", 1.0)),
            tree_method="hist",
            grow_policy="lossguide",
            max_depth=0,
            objective="reg:quantileerror",
            random_state=42,
            n_jobs=-1,
        )

        fold_scores = []
        fold_best_iter = []

        for (tr, te) in folds:
            dtr = df.iloc[tr]
            dte = df.iloc[te]

            Xtr = dtr[predictors]
            Xte = dte[predictors]

            ytr_log = np.log(dtr[response_col].to_numpy(dtype=float))
            yte_log = np.log(dte[response_col].to_numpy(dtype=float))
            yte_exc = dte[y_exc_col].to_numpy(dtype=int)

            # Fit QXGB on log(y)
            qreg = xgb.XGBRegressor(
                **base_params,
                quantile_alpha=alphas,
                early_stopping_rounds=early_stopping_rounds
            )

            # Early stopping: use first alpha's built-in metric for stopping signal.
            # (Your selection metric is computed manually below.)
            qreg.fit(
                Xtr, ytr_log,
                eval_set=[(Xte, yte_log)],
                verbose=False
            )
            fold_best_iter.append(int(getattr(qreg, "best_iteration", np.nan)))

            # Predict quantiles of log(y)
            Qlog = _to_mat(qreg.predict(Xte), n=len(Xte), K=K)
            Qlog = monotone_in_p(Qlog)

            if score_mode == "pinball":
                # average pinball across quantiles on log scale
                losses = [
                    mean_pinball_loss(yte_log, Qlog[:, j], alpha=alphas[j])
                    for j in range(K)
                ]
                score = float(np.mean(losses))  # lower better

            else:
                # implied exceedance prob at 30 via inversion at log(30)
                p_exc = exceed_prob_from_quantiles(Qlog, ps=alphas, threshold=THRESH_LOG)

                if score_mode == "exceed_logloss":
                    score = float(log_loss(yte_exc, p_exc, labels=[0, 1]))  # lower better
                elif score_mode == "exceed_brier":
                    score = float(brier_score_loss(yte_exc, p_exc))         # lower better
                elif score_mode == "exceed_pr_auc":
                    score = float(average_precision_score(yte_exc, p_exc))  # higher better
                else:
                    raise ValueError(f"Unknown score_mode: {score_mode}")

            fold_scores.append(score)

        # Aggregate across folds
        mean_score = float(np.mean(fold_scores))
        std_score = float(np.std(fold_scores))
        mean_best_iter = float(np.nanmean(fold_best_iter))

        out = dict(
            grid_id=int(gi),
            score_mode=score_mode,
            mean_score=mean_score,
            std_score=std_score,
            mean_best_iteration=mean_best_iter,
        )
        out.update({k: float(v) for k, v in row.to_dict().items()})
        results.append(out)

        if verbose_every and ((gi + 1) % verbose_every == 0):
            print(f"[{gi+1}/{len(tune_grid)}] mode={score_mode} mean_score={mean_score:.6f} "
                  f"best_iter~{mean_best_iter:.1f}")

    results_df = pd.DataFrame(results)

    # Choose best depending on score_mode direction
    if score_mode == "exceed_pr_auc":
        results_df = results_df.sort_values(["mean_score", "std_score"], ascending=[False, True]).reset_index(drop=True)
    else:
        results_df = results_df.sort_values(["mean_score", "std_score"], ascending=[True, True]).reset_index(drop=True)

    best_row = results_df.iloc[0].to_dict()
    return best_row, results_df

In [20]:
tune_alphas = [0.01, 0.1, 0.5, 0.9, 0.99]
tune_alphas

[0.01, 0.1, 0.5, 0.9, 0.99]

In [21]:
tune_grid3 = tune_grid.sample(n=100, random_state=42).reset_index(drop=True)

In [111]:
# ============================================================
# Run tuning (pick one score mode)
# ============================================================
# start_time = time.perf_counter()
# best_row_q, qxgb_results = tune_qxgb_groupcv_earlystop(
#     df=df,
#     predictors=hourly_predictors,
#     tune_grid=tune_grid3,
#     alphas=tune_alphas,
#     response_col="y",
#     group_col="Monitor",
#     y_exc_col="y_exc",
#     n_estimators=2000,
#     early_stopping_rounds=25,
#     n_folds=5,          
#     seed=42,
#     score_mode="exceed_brier",   # or "pinball", "exceed_brier", "exceed_pr_auc"
#     verbose_every=1,
# )
# end_time = time.perf_counter()

# execution_time = end_time - start_time
# print(f"Execution time: {execution_time:.4f} seconds")

# print("\nBest QXGB row:")
# print(best_row_q)

# # Save results
# qxgb_results.toparquet(save_path + "qxgb_cv_results.parquet", index=False)

[1/100] mode=exceed_brier mean_score=0.033076 best_iter~45.8
[2/100] mode=exceed_brier mean_score=0.031886 best_iter~32.6
[3/100] mode=exceed_brier mean_score=0.033234 best_iter~74.2
[4/100] mode=exceed_brier mean_score=0.033418 best_iter~121.2
[5/100] mode=exceed_brier mean_score=0.033301 best_iter~55.0
[6/100] mode=exceed_brier mean_score=0.032266 best_iter~35.2
[7/100] mode=exceed_brier mean_score=0.033258 best_iter~34.2
[8/100] mode=exceed_brier mean_score=0.032644 best_iter~18.2
[9/100] mode=exceed_brier mean_score=0.033257 best_iter~28.8
[10/100] mode=exceed_brier mean_score=0.033522 best_iter~31.2
[11/100] mode=exceed_brier mean_score=0.033318 best_iter~159.0
[12/100] mode=exceed_brier mean_score=0.033973 best_iter~73.4
[13/100] mode=exceed_brier mean_score=0.033382 best_iter~53.4
[14/100] mode=exceed_brier mean_score=0.032790 best_iter~63.2
[15/100] mode=exceed_brier mean_score=0.032882 best_iter~93.4
[16/100] mode=exceed_brier mean_score=0.033117 best_iter~107.8
[17/100] mode=

6347.0863 seconds to run

In [14]:
qxgb_cv_results = pd.read_parquet(save_path +"qxgb_cv_results.parquet")

best_row_qxgb = qxgb_cv_results.iloc[0].to_dict()
print("\nBest row (by mean brier, then std):")
print(best_row_qxgb)


Best row (by mean brier, then std):
{'grid_id': 55, 'score_mode': 'exceed_brier', 'mean_score': 0.031055496661443888, 'std_score': 0.05997298025254908, 'mean_best_iteration': 23.4, 'max_leaves': 256.0, 'eta': 0.2, 'gamma': 0.01, 'colsample_bytree': 0.5, 'min_child_weight': 5.0, 'subsample': 0.5, 'reg_lambda': 5.0}


In [23]:
qxgb_cv_results

,grid_id,score_mode,mean_score,std_score,mean_best_iteration,max_leaves,eta,gamma,colsample_bytree,min_child_weight,subsample,reg_lambda
0,55,exceed_brier,0.031055,0.059973,23.4,256.0,0.20,0.01,0.50,5.0,0.50,5.0
1,70,exceed_brier,0.031246,0.060340,56.2,64.0,0.20,0.10,0.25,10.0,0.50,5.0
2,1,exceed_brier,0.031886,0.061508,32.6,128.0,0.20,0.10,0.25,10.0,0.25,50.0
3,87,exceed_brier,0.031964,0.061719,30.8,64.0,0.20,0.01,0.50,5.0,0.75,5.0
4,64,exceed_brier,0.032006,0.061880,29.2,128.0,0.20,0.10,0.50,5.0,0.50,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...
95,9,exceed_brier,0.033522,0.064808,31.2,256.0,0.10,0.01,0.75,5.0,0.75,20.0
96,93,exceed_brier,0.033627,0.065019,141.0,256.0,0.05,1.00,0.25,1.0,0.25,20.0
97,77,exceed_brier,0.033654,0.065076,15.2,256.0,0.20,0.01,0.25,5.0,0.50,20.0
98,81,exceed_brier,0.033690,0.065124,15.0,256.0,0.20,0.01,0.25,5.0,0.50,5.0


### Set new params

In [15]:
# params for binary exceedance model
bin_params_new = build_xgb_params_from_best_row(
    best_row_bin,
    objective="binary:logistic",
    eval_metric="logloss"
)

# params for quantile model (pinball loss)
# NOTE: objective will be set in the regressor constructor; eval_metric can be omitted or set.
q_params_new = build_xgb_params_from_best_row(
    best_row_qxgb,
    objective="reg:quantileerror",
    eval_metric=pinball_loss_sklearn
)

In [16]:
bin_params_new

{'max_leaves': 64,
 'learning_rate': 0.2,
 'gamma': 0.1,
 'colsample_bytree': 0.5,
 'min_child_weight': 5.0,
 'subsample': 0.75,
 'tree_method': 'hist',
 'grow_policy': 'lossguide',
 'objective': 'binary:logistic',
 'random_state': 42,
 'n_estimators': 204,
 'reg_lambda': 50,
 'eval_metric': 'logloss'}

In [17]:
q_params_new

{'max_leaves': 256,
 'learning_rate': 0.2,
 'gamma': 0.01,
 'colsample_bytree': 0.5,
 'min_child_weight': 5.0,
 'subsample': 0.5,
 'tree_method': 'hist',
 'grow_policy': 'lossguide',
 'objective': 'reg:quantileerror',
 'random_state': 42,
 'n_estimators': 23,
 'reg_lambda': 5,
 'custom_metric': <function __main__.pinball_loss_sklearn(preds, dtrain)>}

## Spatial test

### Helper functions

In [15]:
loso = LeaveOneGroupOut()

def loso_binary_with_lag_log(df, predictors, loso, bin_params, *, time_col, verbose=True):
    metrics_rows = []
    pred_parts = []

    for fold, (tr, te) in enumerate(
        loso.split(df, df["y_exc"].to_numpy(), groups=df["site_id"].to_numpy()),
        start=1
    ):
        train = df.iloc[tr].copy()
        test  = df.iloc[te].copy()
        site_id = str(test["site_id"].iloc[0])

        # LOG-scale lag
        train = add_lag_columns_train_log(train, time_col=time_col, z_col="z")
        test  = add_lag_column_test_log(test, train, time_col=time_col, z_col="z")

        train = train.dropna(subset=["lag"])
        test  = test.dropna(subset=["lag"])

        Xtr = train[predictors + ["lag"]]
        ytr = train["y_exc"].to_numpy()
        Xte = test[predictors + ["lag"]]
        yte = test["y_exc"].to_numpy()

        # imbalance weight
        pos = max(int(ytr.sum()), 1)
        neg = max(int(len(ytr) - ytr.sum()), 1)
        scale_pos_weight = neg / pos

        clf = xgb.XGBClassifier(**bin_params, scale_pos_weight=scale_pos_weight)
        clf.fit(Xtr, ytr)
        p = clf.predict_proba(Xte)[:, 1]

        # ---- store per-row predictions for this fold ----
        pred_fold = test[[time_col, "mon_utm_x", "mon_utm_y", "Monitor"]].copy()
        pred_fold["obs"] = test["y"].to_numpy()  # observed ppb
        pred_fold["pred_exceedance_probability"] = p
        pred_fold["fold"] = fold
        pred_fold["site"] = site_id
        pred_parts.append(pred_fold)

        # ---- fold metrics ----
        row = dict(
            fold=fold, site=site_id,
            n_test=len(yte),
            pos_rate_test=float(yte.mean()),
            brier=brier_score_loss(yte, p),
            logloss=log_loss(yte, p, labels=[0, 1]),
        )
        if len(np.unique(yte)) == 2:
            row["roc_auc"] = roc_auc_score(yte, p)
            row["pr_auc"]  = average_precision_score(yte, p)
        else:
            row["roc_auc"] = np.nan
            row["pr_auc"]  = np.nan

        metrics_rows.append(row)

        if verbose:
            print(
                f"[LOSO Binary(log-lag)] fold={fold:02d} site={site_id} "
                f"n_test={row['n_test']} pos_rate={row['pos_rate_test']:.4f} "
                f"brier={row['brier']:.5f} logloss={row['logloss']:.5f} "
                f"roc_auc={row['roc_auc'] if not np.isnan(row['roc_auc']) else 'NA'} "
                f"pr_auc={row['pr_auc'] if not np.isnan(row['pr_auc']) else 'NA'}"
            )

    df_metrics = pd.DataFrame(metrics_rows)
    df_pred = pd.concat(pred_parts, ignore_index=True) if pred_parts else pd.DataFrame()

    return df_metrics, df_pred


def loso_quantile_residual_log(df, predictors, alphas, loso, q_params,
                               *, time_col, verbose=True):
    metrics_rows = []
    pred_parts = []

    for fold, (tr, te) in enumerate(
        loso.split(df, df["z"].to_numpy(), groups=df["site_id"].to_numpy()),
        start=1
    ):
        train = df.iloc[tr].copy()
        test  = df.iloc[te].copy()
        site_id = str(test["site_id"].iloc[0])

        # LOG-scale lag
        train = add_lag_columns_train_log(train, time_col=time_col, z_col="z")
        test  = add_lag_column_test_log(test, train, time_col=time_col, z_col="z")

        train = train.dropna(subset=["lag"]).copy()
        test  = test.dropna(subset=["lag"]).copy()

        # residual on log scale: r = log(y) - lag
        train["r"] = train["z"] - train["lag"]
        test["r"]  = test["z"]  - test["lag"]

        Xtr = train[predictors + ["lag"]]
        ytr = train["r"].to_numpy()
        Xte = test[predictors + ["lag"]]
        yte_r   = test["r"].to_numpy()
        yte_exc = test["y_exc"].to_numpy()

        qreg = xgb.XGBRegressor(
            **q_params,
            quantile_alpha=alphas,
        )
        qreg.fit(Xtr, ytr)

        # Residual quantiles on log-residual scale
        Qr = _to_mat(qreg.predict(Xte), n=len(Xte), K=len(alphas))

        # Reconstruct quantiles of Z = log(Y): Qz(p) = lag + Qr(p)
        Qz = Qr + test["lag"].to_numpy().reshape(-1, 1)

        # Pinball on residuals (log scale)
        pinballs = [mean_pinball_loss(yte_r, Qr[:, j], alpha=alphas[j]) for j in range(len(alphas))]
        avg_pinball = float(np.mean(pinballs))

        # Exceedance for Y>30 is Z>log(30)
        p_exc = exceed_prob_from_quantiles(Qz, ps=alphas, threshold=THRESH_LOG)

        # ---- store per-row predictions for this fold ----
        pred_fold = test[[time_col, "mon_utm_x", "mon_utm_y", "Monitor"]].copy()
        pred_fold["obs"] = test["y"].to_numpy()  # observed ppb
        pred_fold["pred_exceedance_probability"] = p_exc
        pred_fold["fold"] = fold
        pred_fold["site"] = site_id
        pred_parts.append(pred_fold)

        # ---- fold metrics ----
        row = dict(
            fold=fold, site=site_id,
            n_test=len(yte_exc),
            pos_rate_test=float(yte_exc.mean()),
            avg_pinball_resid=avg_pinball,
            brier_exc=brier_score_loss(yte_exc, p_exc),
            logloss_exc=log_loss(yte_exc, p_exc, labels=[0, 1]),
        )
        if len(np.unique(yte_exc)) == 2:
            row["roc_auc_exc"] = roc_auc_score(yte_exc, p_exc)
            row["pr_auc_exc"]  = average_precision_score(yte_exc, p_exc)
        else:
            row["roc_auc_exc"] = np.nan
            row["pr_auc_exc"]  = np.nan

        metrics_rows.append(row)

        if verbose:
            print(
                f"[LOSO Quantile(log-resid)] fold={fold:02d} site={site_id} "
                f"n_test={row['n_test']} pos_rate={row['pos_rate_test']:.4f} "
                f"avg_pinball_resid={row['avg_pinball_resid']:.5f} "
                f"brier_exc={row['brier_exc']:.5f} logloss_exc={row['logloss_exc']:.5f} "
                f"roc_auc_exc={row['roc_auc_exc'] if not np.isnan(row['roc_auc_exc']) else 'NA'} "
                f"pr_auc_exc={row['pr_auc_exc'] if not np.isnan(row['pr_auc_exc']) else 'NA'}"
            )

    df_metrics = pd.DataFrame(metrics_rows)
    df_pred = pd.concat(pred_parts, ignore_index=True) if pred_parts else pd.DataFrame()

    return df_metrics, df_pred

In [16]:
def _make_site_key(df, x_col="mon_utm_x", y_col="mon_utm_y"):
    return df[[x_col, y_col]].astype(str).agg("_".join, axis=1)

def make_monitor_loso_summary(
    monitor_df, df_bin_lag, df_q_resid,
    monitor_name_col="Monitor",
    x_col="mon_utm_x", y_col="mon_utm_y",
    metrics=None,                 # <- choose which metric columns to return
    include_checks=True           # <- include n_match/prop_match
):
    """
    Build a per-monitor LOSO summary and add:
      - baseline_brier = r(1-r) using prop_exceed (r)
      - BSS for each model:
            BSS = 1 - (Brier_model / baseline_brier)
        (computed for binary and quantile exceedance probs)
    Parameters
    ----------
    metrics : None or list[str]
        If None, returns a sensible default set of columns.
        If list, returns the requested metrics (plus identifiers + core counts).
        Allowed metric names (strings):
          "bin_brier","bin_logloss","bin_roc_auc","bin_pr_auc",
          "q_avg_pinball_resid","q_brier","q_logloss","q_roc_auc","q_pr_auc",
          "baseline_brier","bin_bss","q_bss",
          "n_match","prop_match"
    """

    # --- monitor lookup with site key ---
    mon = monitor_df.copy()
    mon["site"] = _make_site_key(mon, x_col=x_col, y_col=y_col)

    # --- binary LOSO metrics (one row per site/fold) ---
    bin_df = df_bin_lag.copy()
    bin_df["n_exceed"] = np.round(bin_df["pos_rate_test"] * bin_df["n_test"]).astype(int)

    bin_keep = bin_df[[
        "site", "n_test", "n_exceed", "pos_rate_test",
        "brier", "logloss", "roc_auc", "pr_auc"
    ]].rename(columns={
        "n_test": "n",
        "pos_rate_test": "prop_exceed",
        "brier": "bin_brier",
        "logloss": "bin_logloss",
        "roc_auc": "bin_roc_auc",
        "pr_auc": "bin_pr_auc",
    })

    # --- quantile LOSO metrics (one row per site/fold) ---
    q_df = df_q_resid.copy()
    q_df["n_exceed_q"] = np.round(q_df["pos_rate_test"] * q_df["n_test"]).astype(int)

    q_keep = q_df[[
        "site",
        "avg_pinball_resid", "brier_exc", "logloss_exc", "roc_auc_exc", "pr_auc_exc",
        "n_test", "pos_rate_test", "n_exceed_q"
    ]].rename(columns={
        "avg_pinball_resid": "q_avg_pinball_resid",
        "brier_exc": "q_brier",
        "logloss_exc": "q_logloss",
        "roc_auc_exc": "q_roc_auc",
        "pr_auc_exc": "q_pr_auc",
        "n_test": "n_q",
        "pos_rate_test": "prop_exceed_q",
        "n_exceed_q": "n_exceed_q",
    })

    # --- merge everything ---
    out = (
        mon.merge(bin_keep, on="site", how="left")
           .merge(q_keep, on="site", how="left")
    )

    # --- baseline Brier using the observed exceedance rate r = prop_exceed ---
    r = out["prop_exceed"].astype(float)
    out["baseline_brier"] = r * (1.0 - r)

    # --- Brier Skill Score (BSS) for each model ---
    # BSS = 1 - (Brier_model / baseline_brier)
    # handle baseline_brier==0 (all 0 or all 1): set BSS to NaN
    denom = out["baseline_brier"].to_numpy()
    denom_safe = np.where((denom > 0) & np.isfinite(denom), denom, np.nan)

    out["bin_bss"] = 1.0 - (out["bin_brier"].to_numpy() / denom_safe)
    out["q_bss"]   = 1.0 - (out["q_brier"].to_numpy() / denom_safe)

    # Optional: show whether model beats baseline directly
    out["bin_brier_better_than_baseline"] = out["bin_brier"] < out["baseline_brier"]
    out["q_brier_better_than_baseline"]   = out["q_brier"] < out["baseline_brier"]

    # --- Optional consistency checks ---
    if include_checks:
        out["n_match"] = (out["n"].astype("float") == out["n_q"].astype("float"))
        out["prop_match"] = np.isclose(
            out["prop_exceed"].astype("float"),
            out["prop_exceed_q"].astype("float"),
            equal_nan=True
        )

    # --- default columns / user-selected metric columns ---
    id_cols = [monitor_name_col]
    core_cols = ["n", "n_exceed", "prop_exceed", "baseline_brier"]

    default_metrics = [
        "bin_brier", "bin_bss", "bin_logloss", "bin_roc_auc", "bin_pr_auc",
        "q_brier", "q_bss", "q_logloss", "q_roc_auc", "q_pr_auc",
        "q_avg_pinball_resid",
        "bin_brier_better_than_baseline", "q_brier_better_than_baseline",
    ]
    if include_checks:
        default_metrics += ["n_match", "prop_match"]

    if metrics is None:
        metrics = default_metrics

    # Ensure only existing columns are requested
    wanted = id_cols + core_cols + metrics
    wanted = [c for c in wanted if c in out.columns]
    out = out[wanted]

    # Warn if any monitors didn't match a fold (site key mismatch)
    missing = out[out["n"].isna() & out.get("q_brier", pd.Series(np.nan, index=out.index)).isna()]
    if len(missing) > 0:
        print("WARNING: Some monitors did not match any LOSO fold 'site' key (check site_id construction).")
        print(missing[[monitor_name_col, x_col, y_col]].head(20))

    return out.sort_values(monitor_name_col)

In [17]:
def count_in_alpha_intervals(df, col, alphas):
    """
    Given alphas (prob levels) and a numeric column in df, count how many values
    fall into each interval formed by the alphas.

    Intervals are:
      (-inf, a1], (a1, a2], ..., (a_{K-1}, aK], (aK, +inf)

    Returns a DataFrame with interval labels and counts.
    """
    alphas = np.asarray(sorted(set(alphas)), dtype=float)
    x = df[col].to_numpy(dtype=float)
    x = 1 - x
    
    bins = np.concatenate(([-np.inf], alphas, [np.inf]))

    labels = []
    labels.append(f"(-inf, {alphas[0]}]")
    for i in range(len(alphas) - 1):
        labels.append(f"({alphas[i]}, {alphas[i+1]}]")
    labels.append(f"({alphas[-1]}, inf)")

    cats = pd.cut(x, bins=bins, right=True, include_lowest=True, labels=labels)

    # value_counts without sort kwarg, then reindex to preserve interval order
    vc = cats.value_counts(dropna=False)
    vc = vc.reindex(labels, fill_value=0)

    out = vc.reset_index()
    out.columns = ["interval", "count"]
    return out

In [64]:
def expected_exceedances_by_group(df_pred, *,
                                   group_col="Monitor",
                                   obs_col="obs",
                                   p_col="pred_exceedance_probability"):
    """
    Group by monitor, compute:
      - n
      - avg_pred_p
      - expected_exceed
      - observed_exceed
      - observed_rate
    """
    df = df_pred.copy()
    df["_exc_obs"] = (df[obs_col].to_numpy(dtype=float) > float(THRESH_PPB)).astype(int)
    
    out = (
        df.groupby(group_col, as_index=False)
          .agg(
              n=(obs_col, "size"),
              observed_exceed=("_exc_obs", "sum"),
              expected_exceed=(p_col, lambda s: float(s.mean()) * s.size),
          )
    )
    out["expected_exceed"] = np.rint(out["expected_exceed"]).astype(int)

    # Overall row (pooled over all observations)
    n_group = np.rint(out["n"].mean()).astype(int)              

    mean = pd.DataFrame([{
        group_col: "Mean",
        "n": np.rint(out["n"].mean()).astype(int)  ,
        "observed_exceed": np.rint(out["observed_exceed"].mean()).astype(int)    ,
        "expected_exceed": np.rint(out["expected_exceed"].mean()).astype(int)       
    }])

    sums = pd.DataFrame([{
        group_col: "Sum",
        "n": np.rint(out["n"].sum()).astype(int)  ,
        "observed_exceed": np.rint(out["observed_exceed"].sum()).astype(int)    ,
        "expected_exceed": np.rint(out["expected_exceed"].sum()).astype(int)       
    }])

    out = pd.concat([out, mean, sums], ignore_index=True)
    return out

### New tunings

In [18]:
df_bin_lag_metrics_newtuning, df_bin_lag_pred_newtuning = loso_binary_with_lag_log(
    df, hourly_predictors, loso, bin_params_new, time_col=time_col, verbose=True
)

print("\n=== LOSO | Binary logistic (log-lag) | fold metrics ===")
print(df_bin_lag_metrics_newtuning.mean(numeric_only=True).round(4))

[LOSO Binary(log-lag)] fold=01 site=369852.5359163884_3754053.0239289817 n_test=29467 pos_rate=0.0001 brier=0.00007 logloss=0.00089 roc_auc=0.5660105209570677 pr_auc=0.00011513599904261696
[LOSO Binary(log-lag)] fold=02 site=369932.10194513993_3754245.360266213 n_test=5279 pos_rate=0.0000 brier=0.00000 logloss=0.00009 roc_auc=NA pr_auc=NA
[LOSO Binary(log-lag)] fold=03 site=370408.9455206478_3750865.96396696 n_test=30121 pos_rate=0.0000 brier=0.00002 logloss=0.00011 roc_auc=NA pr_auc=NA
[LOSO Binary(log-lag)] fold=04 site=373408.60980845685_3746402.1236518873 n_test=20070 pos_rate=0.0000 brier=0.00021 logloss=0.00089 roc_auc=NA pr_auc=NA
[LOSO Binary(log-lag)] fold=05 site=376372.05412800366_3748307.7286123442 n_test=20257 pos_rate=0.0002 brier=0.00025 logloss=0.00115 roc_auc=0.9989878042759097 pr_auc=0.10364422084623323
[LOSO Binary(log-lag)] fold=06 site=376793.2331856196_3745266.662266282 n_test=32216 pos_rate=0.0001 brier=0.00019 logloss=0.00073 roc_auc=0.998592700669502 pr_auc=0.0

In [19]:
df_bin_lag_metrics_newtuning.toparquet(save_path + "loso_binary_metrics_newtuning.parquet", index=False)
df_bin_lag_pred_newtuning.toparquet(save_path + "loso_binary_predictions_newtuning.parquet", index=False)

In [20]:
alphas = [0.001, 0.01, 0.03, 0.05, 0.075, 
          0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9,
          0.925, 0.95, 0.97, 0.99, 0.999, 0.9999, 0.99999, 0.999999, 0.9999999, 0.99999999, 0.999999999, 0.9999999999]

df_q_resid_metrics_newtuning, df_q_resid_pred_newtuning = loso_quantile_residual_log(
    df, hourly_predictors, alphas, loso, q_params_new, time_col=time_col, verbose=True
)

print("\n=== LOSO | Quantile residual (log-resid) | fold metrics ===")
print(df_q_resid_metrics_newtuning .mean(numeric_only=True).round(4))

[LOSO Quantile(log-resid)] fold=01 site=369852.5359163884_3754053.0239289817 n_test=29467 pos_rate=0.0001 avg_pinball_resid=0.15247 brier_exc=0.00007 logloss_exc=0.00167 roc_auc_exc=0.42548786696080093 pr_auc_exc=6.787253537855906e-05
[LOSO Quantile(log-resid)] fold=02 site=369932.10194513993_3754245.360266213 n_test=5279 pos_rate=0.0000 avg_pinball_resid=0.16223 brier_exc=0.00000 logloss_exc=0.00002 roc_auc_exc=NA pr_auc_exc=NA
[LOSO Quantile(log-resid)] fold=03 site=370408.9455206478_3750865.96396696 n_test=30121 pos_rate=0.0000 avg_pinball_resid=0.13119 brier_exc=0.00000 logloss_exc=0.00012 roc_auc_exc=NA pr_auc_exc=NA
[LOSO Quantile(log-resid)] fold=04 site=373408.60980845685_3746402.1236518873 n_test=20070 pos_rate=0.0000 avg_pinball_resid=0.22672 brier_exc=0.00134 logloss_exc=0.00659 roc_auc_exc=NA pr_auc_exc=NA
[LOSO Quantile(log-resid)] fold=05 site=376372.05412800366_3748307.7286123442 n_test=20257 pos_rate=0.0002 avg_pinball_resid=0.16151 brier_exc=0.00019 logloss_exc=0.00106

In [21]:
df_q_resid_metrics_newtuning.toparquet(save_path + "loso_quantile_metrics_newtuning.parquet", index=False)
df_q_resid_pred_newtuning.toparquet(save_path + "loso_quantile_predictions_newtuning.parquet", index=False)

In [18]:
df_bin_metrics_newtuning = pd.read_parquet(save_path +"loso_binary_metrics_newtuning.parquet")
df_bin_pred_newtuning    = pd.read_parquet(save_path +"loso_binary_predictions_newtuning.parquet")

df_q_metrics_newtuning   = pd.read_parquet(save_path +"loso_quantile_metrics_newtuning.parquet")
df_q_pred_newtuning      = pd.read_parquet(save_path +"loso_quantile_predictions_newtuning.parquet")

In [23]:
# the interval represents the predicted quantiles that contain 30ppb
# 1-interval is the interval of exceedance probability
# so 249272 observations have 0.0000001*100% to 0.000001*100% exceedance probability
count_in_alpha_intervals(df_q_pred_newtuning, "pred_exceedance_probability", alphas)

,interval,count
0,"(-inf, 0.001]",0
1,"(0.001, 0.01]",0
2,"(0.01, 0.03]",0
3,"(0.03, 0.05]",0
4,"(0.05, 0.075]",0
5,"(0.075, 0.1]",0
6,"(0.1, 0.2]",0
7,"(0.2, 0.3]",9
8,"(0.3, 0.4]",28
9,"(0.4, 0.5]",57


In [24]:
make_monitor_loso_summary(monitor_coord_map, df_bin_metrics_newtuning, df_q_metrics_newtuning,
                          metrics = ["bin_brier", "bin_bss", "bin_roc_auc", "bin_pr_auc",
                                     "q_brier",  "q_bss", "q_roc_auc", "q_pr_auc"])

,Monitor,n,n_exceed,prop_exceed,baseline_brier,bin_brier,bin_bss,bin_roc_auc,bin_pr_auc,q_brier,q_bss,q_roc_auc,q_pr_auc
0,Chico,2521,428,0.169774,0.140951,0.143658,-0.019208,0.828752,0.598586,1.594769e-01,-0.131437,0.601929,0.430886
1,ElSegundo,5279,0,0.000000,0.000000,0.000004,NaN,NaN,NaN,1.161090e-09,NaN,NaN,NaN
2,ElmAve,32216,3,0.000093,0.000093,0.000194,-1.085600,0.998593,0.042183,8.344773e-05,0.103799,0.999565,0.468835
3,FirstMethodist,30759,12,0.000390,0.000390,0.000279,0.285684,0.998585,0.492148,3.380530e-04,0.133147,0.988354,0.386556
4,GStreet,22217,2,0.000090,0.000090,0.001435,-14.944564,0.958249,0.071969,1.517701e-04,-0.686090,0.839343,0.027918
5,GuenserPark,27326,18,0.000659,0.000658,0.000588,0.106438,0.997868,0.364733,6.085684e-04,0.075517,0.996930,0.147828
6,HarborPark,26127,6,0.000230,0.000230,0.000790,-2.440545,0.997639,0.104817,2.278927e-04,0.007413,0.997531,0.213884
7,Hudson,32091,12,0.000374,0.000374,0.000712,-0.905559,0.993776,0.482531,3.487379e-04,0.067039,0.976784,0.150029
8,InnerPort,29107,8,0.000275,0.000275,0.000196,0.286314,0.919430,0.483632,2.422373e-03,-7.815923,0.958873,0.093007
9,Judson,30014,107,0.003565,0.003552,0.004963,-0.397131,0.997141,0.513849,2.432155e-03,0.315328,0.946586,0.614844


For binary classifier:
- Model is useful for 3 sites (BSS > 0)
 
For QXGB:
- Model is useful for 7 sites

In [65]:
exp_bin = expected_exceedances_by_group(df_bin_pred_newtuning, group_col = "Monitor")
exp_q   = expected_exceedances_by_group(df_q_pred_newtuning, group_col = "Monitor")
exp_bin.merge(exp_q, on = ['Monitor', 'n'], how = 'left', suffixes = ('_bin', '_q'))

,Monitor,n,observed_exceed_bin,expected_exceed_bin,observed_exceed_q,expected_exceed_q
0,Chico,2521,428,76,428,17
1,ElSegundo,5279,0,0,0,0
2,ElmAve,32216,3,9,3,9
3,FirstMethodist,30759,12,10,12,8
4,GStreet,22217,2,51,2,16
5,GuenserPark,27326,18,26,18,9
6,HarborPark,26127,6,31,6,9
7,Hudson,32091,12,48,12,21
8,InnerPort,29107,8,18,8,310
9,Judson,30014,107,308,107,39


## 8020 tests

### Helpers

In [82]:
# -----------------------------
# 80/20 RANDOM SPLIT (row-wise)
# -----------------------------
def random8020_binary_log(df, predictors, bin_params, *, time_col, test_size=0.2, seed=42, verbose=True):
    """
    One random 80/20 split (row-wise)
    Returns (df_metrics, df_pred) similar to LOSO versions.
    """
    tr_idx, te_idx = train_test_split(np.arange(len(df)), test_size=test_size, random_state=seed, shuffle=True)

    train = df.iloc[tr_idx].copy()
    test  = df.iloc[te_idx].copy()

    Xtr = train[predictors]
    ytr = train["y_exc"].to_numpy()
    Xte = test[predictors]
    yte = test["y_exc"].to_numpy()

    # imbalance weight
    pos = max(int(ytr.sum()), 1)
    neg = max(int(len(ytr) - ytr.sum()), 1)
    scale_pos_weight = neg / pos

    clf = xgb.XGBClassifier(**bin_params, scale_pos_weight=scale_pos_weight)
    clf.fit(Xtr, ytr)
    p = clf.predict_proba(Xte)[:, 1]

    # per-row predictions table
    pred_df = test[[time_col, "mon_utm_x", "mon_utm_y", "Monitor"]].copy()
    pred_df["obs"] = test["y"].to_numpy()
    pred_df["pred_exceedance_probability"] = p
    pred_df["fold"] = 1
    pred_df["site"] = "random_80_20"

    # metrics (single "fold")
    row = dict(
        fold=1,
        site="random_80_20",
        n_test=len(yte),
        pos_rate_test=float(yte.mean()),
        brier=brier_score_loss(yte, p),
        logloss=log_loss(yte, p, labels=[0, 1]),
    )
    if len(np.unique(yte)) == 2:
        row["roc_auc"] = roc_auc_score(yte, p)
        row["pr_auc"]  = average_precision_score(yte, p)
    else:
        row["roc_auc"] = np.nan
        row["pr_auc"]  = np.nan

    if verbose:
        print(
            f"[Random 80/20 Binary(log-lag)] "
            f"n_test={row['n_test']} pos_rate={row['pos_rate_test']:.4f} "
            f"brier={row['brier']:.5f} logloss={row['logloss']:.5f} "
            f"roc_auc={row['roc_auc'] if not np.isnan(row['roc_auc']) else 'NA'} "
            f"pr_auc={row['pr_auc'] if not np.isnan(row['pr_auc']) else 'NA'}"
        )

    return pd.DataFrame([row]), pred_df


def random8020_quantile_residual_log(df, predictors, alphas, q_params, *, time_col,
                                     test_size=0.2, seed=42, verbose=True):
    """
    One random 80/20 split (row-wise), quantile model
    Returns (df_metrics, df_pred) similar to LOSO versions.
    """
    tr_idx, te_idx = train_test_split(np.arange(len(df)), test_size=test_size, random_state=seed, shuffle=True)

    train = df.iloc[tr_idx].copy()
    test  = df.iloc[te_idx].copy()

    Xtr = train[predictors]
    ytr = train["z"].to_numpy()
    Xte = test[predictors]
    yte_r   = test["z"].to_numpy()
    yte_exc = test["y_exc"].to_numpy()

    qreg = xgb.XGBRegressor(**q_params, quantile_alpha=alphas)
    qreg.fit(Xtr, ytr)

    Qz = _to_mat(qreg.predict(Xte), n=len(Xte), K=len(alphas))

    # pinball
    pinballs = [mean_pinball_loss(yte_r, Qz[:, j], alpha=alphas[j]) for j in range(len(alphas))]
    avg_pinball = float(np.mean(pinballs))

    # exceedance prob at log(30)
    p_exc = exceed_prob_from_quantiles(Qz, ps=alphas, threshold=THRESH_LOG)

    pred_df = test[[time_col, "mon_utm_x", "mon_utm_y", "Monitor"]].copy()
    pred_df["obs"] = test["y"].to_numpy()
    pred_df["pred_exceedance_probability"] = p_exc
    pred_df["fold"] = 1
    pred_df["site"] = "random_80_20"

    row = dict(
        fold=1,
        site="random_80_20",
        n_test=len(yte_exc),
        pos_rate_test=float(yte_exc.mean()),
        avg_pinball_resid=avg_pinball,
        brier_exc=brier_score_loss(yte_exc, p_exc),
        logloss_exc=log_loss(yte_exc, p_exc, labels=[0, 1]),
    )
    if len(np.unique(yte_exc)) == 2:
        row["roc_auc_exc"] = roc_auc_score(yte_exc, p_exc)
        row["pr_auc_exc"]  = average_precision_score(yte_exc, p_exc)
    else:
        row["roc_auc_exc"] = np.nan
        row["pr_auc_exc"]  = np.nan

    if verbose:
        print(
            f"[Random 80/20 Quantile(log-resid)] "
            f"n_test={row['n_test']} pos_rate={row['pos_rate_test']:.4f} "
            f"avg_pinball_resid={row['avg_pinball_resid']:.5f} "
            f"brier_exc={row['brier_exc']:.5f} logloss_exc={row['logloss_exc']:.5f} "
            f"roc_auc_exc={row['roc_auc_exc'] if not np.isnan(row['roc_auc_exc']) else 'NA'} "
            f"pr_auc_exc={row['pr_auc_exc'] if not np.isnan(row['pr_auc_exc']) else 'NA'}"
        )

    return pd.DataFrame([row]), pred_df

In [79]:
def make_8020_summary_df(
    bin_metrics_df: pd.DataFrame,
    q_metrics_df: pd.DataFrame,
    *,
    include_flags: bool = True,
    metrics: list | None = None,
):
    """
    Build a multi-row 80/20 summary from many seeds.
    Inputs must include column 'seed'.
    Returns a DataFrame with one row per seed plus a final Mean row (seed='Mean').

    Uses:
      r = pos_rate_test (from binary metrics)
      baseline_brier = r(1-r)
      BSS = 1 - (brier_model / baseline_brier)
    """

    # Merge per-seed (assumes same seeds appear in both)
    m = bin_metrics_df.merge(q_metrics_df, on="seed", suffixes=("_bin", "_q"), how="inner").copy()

    # Pull needed columns (robustly)
    r = m["pos_rate_test_bin"].astype(float)
    m["baseline_brier"] = r * (1.0 - r)

    m["bin_brier"]   = m["brier"].astype(float)
    m["bin_logloss"] = m["logloss"].astype(float)
    m["bin_roc_auc"] = m.get("roc_auc", np.nan)
    m["bin_pr_auc"]  = m.get("pr_auc", np.nan)

    m["q_avg_pinball_resid"] = m.get("avg_pinball_resid", np.nan)
    m["q_brier"]   = m["brier_exc"].astype(float)
    m["q_logloss"] = m["logloss_exc"].astype(float)
    m["q_roc_auc"] = m.get("roc_auc_exc", np.nan)
    m["q_pr_auc"]  = m.get("pr_auc_exc", np.nan)

    # n_test / pos_rate_test
    m["n_test"] = m["n_test_bin"]
    m["pos_rate_test"] = m["pos_rate_test_bin"]

    denom = m["baseline_brier"].to_numpy()
    denom_safe = np.where((denom > 0) & np.isfinite(denom), denom, np.nan)

    m["bin_bss"] = 1.0 - (m["bin_brier"].to_numpy() / denom_safe)
    m["q_bss"]   = 1.0 - (m["q_brier"].to_numpy() / denom_safe)

    if include_flags:
        m["bin_brier_better_than_baseline"] = m["bin_brier"] < m["baseline_brier"]
        m["q_brier_better_than_baseline"]   = m["q_brier"] < m["baseline_brier"]

    # Default column set (same as earlier, plus seed)
    default_cols = [
        "seed", "n_test", "pos_rate_test", "baseline_brier",
        "bin_brier", "bin_bss", "bin_logloss", "bin_roc_auc", "bin_pr_auc",
        "q_brier", "q_bss", "q_logloss", "q_roc_auc", "q_pr_auc",
        "q_avg_pinball_resid",
    ]
    if include_flags:
        default_cols += ["bin_brier_better_than_baseline", "q_brier_better_than_baseline"]

    if metrics is None:
        cols = default_cols
    else:
        base_cols = ["seed", "n_test", "pos_rate_test", "baseline_brier"]
        cols = base_cols + metrics
        cols = [c for c in cols if c in m.columns]

    out = m[cols].copy()

    # Add Mean row (numeric means; keep booleans as NA)
    mean_row = {"seed": "Mean"}
    for c in out.columns:
        if c == "seed":
            continue
        if pd.api.types.is_bool_dtype(out[c]):
            mean_row[c] = np.nan
        else:
            mean_row[c] = pd.to_numeric(out[c], errors="coerce").mean()

    out = pd.concat([out, pd.DataFrame([mean_row])], ignore_index=True)
    return out

### New tunings

In [67]:
seeds = list(range(1, 21))

In [75]:
# Run 8020 for each seed and store in df
bin_metrics_list = []
q_metrics_list = []
bin_pred_list = []
q_pred_list = []

for s in seeds:
    print("Running seed " + str(s))
    bin_metrics_8020, bin_pred_8020 = random8020_binary_log(
        df, hourly_predictors, bin_params_new, time_col=time_col, test_size=0.2, seed=s, verbose=False
    )
    q_metrics_8020, q_pred_8020 = random8020_quantile_residual_log(
        df, hourly_predictors, alphas, q_params_new, time_col=time_col, test_size=0.2, seed=s, verbose=False
    )

    # attach seed to the one-row metrics
    bm = bin_metrics_8020.copy()
    bm.insert(0, "seed", s)
    qm = q_metrics_8020.copy()
    qm.insert(0, "seed", s)

    bin_metrics_list.append(bm)
    q_metrics_list.append(qm)

    # keep predictions as list of dfs (also add seed column)
    bp = bin_pred_8020.copy()
    bp.insert(0, "seed", s)
    qp = q_pred_8020.copy()
    qp.insert(0, "seed", s)
    bin_pred_list.append(bp)
    q_pred_list.append(qp)

# combined metrics frames (20 rows each)
bin_metrics_8020_df = pd.concat(bin_metrics_list, ignore_index=True)
q_metrics_8020_df   = pd.concat(q_metrics_list, ignore_index=True)

# combined pred frames (20%*20 rows each)
bin_pred_8020_df = pd.concat(bin_pred_list, ignore_index=True)
q_pred_8020_df   = pd.concat(q_pred_list, ignore_index=True)

Running seed 1
Running seed 2
Running seed 3
Running seed 4
Running seed 5
Running seed 6
Running seed 7
Running seed 8
Running seed 9
Running seed 10
Running seed 11
Running seed 12
Running seed 13
Running seed 14
Running seed 15
Running seed 16
Running seed 17
Running seed 18
Running seed 19
Running seed 20


In [76]:
bin_metrics_8020_df.toparquet(save_path + "bin_metrics_8020_df.parquet", index=False)
q_metrics_8020_df.toparquet(save_path + "q_metrics_8020_df.parquet", index=False)

bin_pred_8020_df.toparquet(save_path + "bin_pred_8020_df.parquet", index=False)
q_pred_8020_df.toparquet(save_path + "q_pred_8020_df.parquet", index=False)

In [77]:
bin_metrics_8020_df_newtuning = pd.read_parquet(save_path +"bin_metrics_8020_df.parquet")
q_metrics_8020_df_newtuning   = pd.read_parquet(save_path +"q_metrics_8020_df.parquet")

bin_pred_8020_df_newtuning = pd.read_parquet(save_path +"bin_pred_8020_df.parquet")
q_pred_8020_df_newtuning   = pd.read_parquet(save_path +"q_pred_8020_df.parquet")

In [80]:
make_8020_summary_df(bin_metrics_8020_df_newtuning, q_metrics_8020_df_newtuning,
                     metrics = ["bin_brier", "bin_bss",
                                "bin_roc_auc", "bin_pr_auc", "q_brier", 
                                "q_bss", "q_roc_auc", "q_pr_auc"])


,seed,n_test,pos_rate_test,baseline_brier,bin_brier,bin_bss,bin_roc_auc,bin_pr_auc,q_brier,q_bss,q_roc_auc,q_pr_auc
0,1,73167.0,0.001695,0.001692,0.000608,0.640417,0.999736,0.900428,0.000646,0.617961,0.995944,0.855088
1,2,73167.0,0.001490,0.001488,0.000984,0.338504,0.990178,0.843368,0.000624,0.580780,0.990497,0.804888
2,3,73167.0,0.001408,0.001406,0.000812,0.422436,0.995913,0.881628,0.000615,0.562222,0.994141,0.766234
3,4,73167.0,0.001640,0.001637,0.000800,0.511152,0.999660,0.883686,0.000686,0.581340,0.999663,0.821763
4,5,73167.0,0.001845,0.001842,0.000795,0.568474,0.999629,0.894841,0.000760,0.587497,0.999444,0.843904
5,6,73167.0,0.001818,0.001814,0.000773,0.574089,0.997029,0.919525,0.000701,0.613606,0.999386,0.853379
6,7,73167.0,0.001490,0.001488,0.000798,0.463591,0.999732,0.912390,0.000567,0.618585,0.999678,0.866722
7,8,73167.0,0.001736,0.001733,0.000817,0.528330,0.999754,0.922579,0.000625,0.639091,0.999694,0.857917
8,9,73167.0,0.001654,0.001651,0.000755,0.542880,0.999840,0.929239,0.000616,0.626700,0.992777,0.854197
9,10,73167.0,0.001749,0.001746,0.000696,0.601243,0.995801,0.940677,0.000600,0.656481,0.999349,0.872695


In [81]:
exp_bin = expected_exceedances_by_group(bin_pred_8020_df_newtuning, group_col = "seed")
exp_q   = expected_exceedances_by_group(q_pred_8020_df_newtuning, group_col = "seed")
exp_bin.merge(exp_q, on = ['seed', 'n'], how = 'left', suffixes = ('_bin', '_q'))

,seed,n,observed_exceed_bin,expected_exceed_bin,observed_exceed_q,expected_exceed_q
0,1,73167,124,180,124,79
1,2,73167,109,192,109,73
2,3,73167,103,173,103,68
3,4,73167,120,191,120,79
4,5,73167,135,202,135,84
5,6,73167,133,196,133,86
6,7,73167,109,183,109,74
7,8,73167,127,202,127,86
8,9,73167,121,194,121,86
9,10,73167,128,200,128,90


## LOEO tests

### Helpers

In [95]:
# -----------------------------
# LOEO SPLIT 
# -----------------------------
def loeo_binary_log(df, predictors, bin_params, *, time_col, seed=42, verbose=True):
    tz = "America/Los_Angeles"
    is_test = df[time_col].between(pd.Timestamp("2021-07-01", tz=tz), pd.Timestamp("2021-08-31", tz=tz), inclusive="both")

    te_idx = np.flatnonzero(is_test.to_numpy())
    tr_idx = np.flatnonzero((~is_test).to_numpy())

    train = df.iloc[tr_idx].copy()
    test  = df.iloc[te_idx].copy()

    Xtr = train[predictors]
    ytr = train["y_exc"].to_numpy()
    Xte = test[predictors]
    yte = test["y_exc"].to_numpy()

    # imbalance weight
    pos = max(int(ytr.sum()), 1)
    neg = max(int(len(ytr) - ytr.sum()), 1)
    scale_pos_weight = neg / pos

    clf = xgb.XGBClassifier(**bin_params, scale_pos_weight=scale_pos_weight)
    clf.fit(Xtr, ytr)
    p = clf.predict_proba(Xte)[:, 1]

    # per-row predictions table
    pred_df = test[[time_col, "mon_utm_x", "mon_utm_y", "Monitor"]].copy()
    pred_df["obs"] = test["y"].to_numpy()
    pred_df["pred_exceedance_probability"] = p
    pred_df["fold"] = 1
    pred_df["site"] = "loeo"

    # metrics (single "fold")
    row = dict(
        fold=1,
        site="loeo",
        n_test=len(yte),
        pos_rate_test=float(yte.mean()),
        brier=brier_score_loss(yte, p),
        logloss=log_loss(yte, p, labels=[0, 1]),
    )
    if len(np.unique(yte)) == 2:
        row["roc_auc"] = roc_auc_score(yte, p)
        row["pr_auc"]  = average_precision_score(yte, p)
    else:
        row["roc_auc"] = np.nan
        row["pr_auc"]  = np.nan

    if verbose:
        print(
            f"[Leave-one-event-out Binary(log-lag)] "
            f"n_test={row['n_test']} pos_rate={row['pos_rate_test']:.4f} "
            f"brier={row['brier']:.5f} logloss={row['logloss']:.5f} "
            f"roc_auc={row['roc_auc'] if not np.isnan(row['roc_auc']) else 'NA'} "
            f"pr_auc={row['pr_auc'] if not np.isnan(row['pr_auc']) else 'NA'}"
        )

    return pd.DataFrame([row]), pred_df


def loeo_quantile_residual_log(df, predictors, alphas, q_params, *, time_col, seed=42, verbose=True):
    tz = "America/Los_Angeles"
    is_test = df[time_col].between(pd.Timestamp("2021-07-01", tz=tz), pd.Timestamp("2021-08-31", tz=tz), inclusive="both")

    te_idx = np.flatnonzero(is_test.to_numpy())
    tr_idx = np.flatnonzero((~is_test).to_numpy())

    train = df.iloc[tr_idx].copy()
    test  = df.iloc[te_idx].copy()

    Xtr = train[predictors]
    ytr = train["z"].to_numpy()
    Xte = test[predictors]
    yte_r   = test["z"].to_numpy()
    yte_exc = test["y_exc"].to_numpy()

    qreg = xgb.XGBRegressor(**q_params, quantile_alpha=alphas)
    qreg.fit(Xtr, ytr)

    Qz = _to_mat(qreg.predict(Xte), n=len(Xte), K=len(alphas))

    # pinball
    pinballs = [mean_pinball_loss(yte_r, Qz[:, j], alpha=alphas[j]) for j in range(len(alphas))]
    avg_pinball = float(np.mean(pinballs))

    # exceedance prob at log(30)
    p_exc = exceed_prob_from_quantiles(Qz, ps=alphas, threshold=THRESH_LOG)

    pred_df = test[[time_col, "mon_utm_x", "mon_utm_y", "Monitor"]].copy()
    pred_df["obs"] = test["y"].to_numpy()
    pred_df["pred_exceedance_probability"] = p_exc
    pred_df["fold"] = 1
    pred_df["site"] = "loeo"

    row = dict(
        fold=1,
        site="loeo",
        n_test=len(yte_exc),
        pos_rate_test=float(yte_exc.mean()),
        avg_pinball_resid=avg_pinball,
        brier_exc=brier_score_loss(yte_exc, p_exc),
        logloss_exc=log_loss(yte_exc, p_exc, labels=[0, 1]),
    )
    if len(np.unique(yte_exc)) == 2:
        row["roc_auc_exc"] = roc_auc_score(yte_exc, p_exc)
        row["pr_auc_exc"]  = average_precision_score(yte_exc, p_exc)
    else:
        row["roc_auc_exc"] = np.nan
        row["pr_auc_exc"]  = np.nan

    if verbose:
        print(
            f"[Leave-one-event-out Quantile(log-resid)] "
            f"n_test={row['n_test']} pos_rate={row['pos_rate_test']:.4f} "
            f"avg_pinball_resid={row['avg_pinball_resid']:.5f} "
            f"brier_exc={row['brier_exc']:.5f} logloss_exc={row['logloss_exc']:.5f} "
            f"roc_auc_exc={row['roc_auc_exc'] if not np.isnan(row['roc_auc_exc']) else 'NA'} "
            f"pr_auc_exc={row['pr_auc_exc'] if not np.isnan(row['pr_auc_exc']) else 'NA'}"
        )

    return pd.DataFrame([row]), pred_df

### New tunings

In [114]:
# Run LOEO
bin_metrics_loeo, bin_pred_loeo = loeo_binary_log(
    df, hourly_predictors, bin_params_new, time_col="day", verbose=False
)
q_metrics_loeo, q_pred_loeo = loeo_quantile_residual_log(
    df, hourly_predictors, alphas, q_params_new, time_col="day", verbose=False
)

In [115]:
q_metrics_loeo.insert(0, "seed", 42)

In [116]:
bin_metrics_loeo.insert(0, "seed", 42)

In [117]:
make_8020_summary_df(bin_metrics_loeo, q_metrics_loeo,
                     metrics = ["bin_brier", "bin_bss",
                                "bin_roc_auc", "bin_pr_auc", "q_brier", 
                                "q_bss", "q_roc_auc", "q_pr_auc"])


,seed,n_test,pos_rate_test,baseline_brier,bin_brier,bin_bss,bin_roc_auc,bin_pr_auc,q_brier,q_bss,q_roc_auc,q_pr_auc
0,42,17835.0,0.0,0.0,1.513194e-09,NaN,NaN,NaN,1.430465e-09,NaN,NaN,NaN
1,Mean,17835.0,0.0,0.0,1.513194e-09,NaN,NaN,NaN,1.430465e-09,NaN,NaN,NaN


In [118]:
exp_bin = expected_exceedances_by_group(bin_pred_loeo, group_col = "site")
exp_q   = expected_exceedances_by_group(q_pred_loeo, group_col = "site")
exp_bin.merge(exp_q, on = ['site', 'n'], how = 'left', suffixes = ('_bin', '_q'))

,site,n,observed_exceed_bin,expected_exceed_bin,observed_exceed_q,expected_exceed_q
0,loeo,17835,0,0,0,1
1,Mean,17835,0,0,0,1
2,Sum,17835,0,0,0,1


In [52]:
# # -----------------------------
# # Fit final models on ALL data and compute per-row exceedance probabilities
# # -----------------------------

# X_final = df[hourly_predictors]
# y_obs = ha_full_full_y[response].to_numpy(dtype=float)
# y_log = np.log(y_obs)
# y_exc_obs = (y_obs > THRESH_PPB).astype(int)

# # ---- (A) Final binary logistic exceedance model: P(Y>30 | X) ----
# final_clf = xgb.XGBClassifier(**bin_params)

# pos = max(int(y_exc_obs.sum()), 1)
# neg = max(int(len(y_exc_obs) - y_exc_obs.sum()), 1)
# final_clf.set_params(scale_pos_weight=neg / pos)

# final_clf.fit(X_final, y_exc_obs)
# p_exc_logistic = final_clf.predict_proba(X_final)[:, 1]

# # ---- (B) Final quantile model on log(Y): quantiles -> invert at log(30) ----
# final_qreg_log = xgb.XGBRegressor(
#     **q_params,
#     quantile_alpha=alphas,
# )
# final_qreg_log.fit(X_final, y_log)

In [53]:
# Qlog = _to_mat(final_qreg_log.predict(X_final), n=len(X_final), K=len(alphas))
# p_exc_quantile_log = exceed_prob_from_quantiles(Qlog, ps=alphas, threshold=THRESH_LOG)

# # ---- Build the output dataframe (keys for merging) ----
# df_pred = df[["dayhour", "mon_utm_x", "mon_utm_y"]].copy()
# df_pred = df_pred.merge(monitor_coord_map, on=["mon_utm_x", "mon_utm_y"], how="left")
# df_pred["y_obs_ppb"] = y_obs
# df_pred["y_exc_obs"] = y_exc_obs
# df_pred["p_exc_logistic"] = p_exc_logistic
# df_pred["p_exc_quantile_log"] = p_exc_quantile_log


# # (Optional) If you have duplicate (dayhour, x, y) rows and want one row per key:
# # df_pred = df_pred.groupby(["dayhour","mon_utm_x","mon_utm_y"], as_index=False).mean()

# # ---- Save to parquet ----
# df_pred.toparquet(save_path + "h2s_exceedance_predictions.parquet", index=False)
# df_pred.to_csv("h2s_exceedance_predictions.csv", index=False)
# df_pred.head()